# **Project Name**     - Airbnb Data Exploration Using SQLite & Pandas (Shamini S)

##### **Project Type**    - SQL_Data_Exploration
##### **Contribution**    - Individual

# **Project Summary -**

**About this notebook**

This notebook uses SQL to explore the Airbnb NYC 2019 listings from inside Python. Two approaches are used: a SQLite database accessed through the `sqlite3` module, and `pandasql`, which runs SQL directly on a pandas DataFrame. It is a companion to the main Airbnb EDA notebook and looks at pricing, availability and neighbourhood patterns using SQL instead of pandas methods.

**What is covered**

* Loading the CSV and storing it as a table in a SQLite database
* Filtering and sorting rows with `WHERE`, `BETWEEN`, `ORDER BY` and `LIMIT`
* Bucketing listings into price bands with `CASE`
* Aggregating with `COUNT`, `AVG` and `ROUND`
* Filtering groups with `GROUP BY` and `HAVING`
* Running SQL on a DataFrame with `pandasql`
* Saving changes and closing the database connection

**Key results**

* Manhattan has the highest average price (\$196.88) and the Bronx the lowest (\$87.50).
* The most expensive listings are priced at about \$10,000 a night.
* *[Add key results from sections 2.2, 3 and 4.2 after running those cells.]*

# **GitHub Link -**

https://github.com/Shamini-tech/Airbnb-Analysis-Capstone-Project

# **Objective**

To use SQL queries to explore the Airbnb NYC 2019 dataset and to practise two ways of running SQL from Python: a SQLite database (`sqlite3`) and `pandasql`.

# ***Let's Begin !***

## ***1. Setting Up the Data***

### Import Libraries, Load Dataset and Preview

The libraries are imported and the Airbnb NYC 2019 CSV is loaded into a pandas DataFrame called `df`.

In [2]:
# Libraries used in this notebook
import pandas as pd
import numpy as np
import sqlite3

# Read the NYC Airbnb listings CSV into a DataFrame
df = pd.read_csv('/content/AB_NYC_2019.csv')

# Quick look at the first five rows
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


## ***2. SQL with SQLite (`sqlite3`)***

### 2.1 Create the Database and Table

A connection to a SQLite database file (`nyc_airbnb.db`) is opened, the DataFrame is saved as a table named `listings` with `to_sql()`, and the table is read back as a check. `if_exists='replace'` overwrites an existing table, and `index=False` keeps the DataFrame index out of the table.

In [3]:
# Open (or create) a local SQLite database file
conn = sqlite3.connect('nyc_airbnb.db')

# Save the DataFrame as a table called 'listings'
# (replace the table if it exists; do not store the DataFrame index as a column)
df.to_sql('listings', conn, if_exists='replace', index=False)

# Read the table back as a quick check
check_df = pd.read_sql_query('SELECT * FROM listings', conn)
check_df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,None,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


**Observation:** The table was created and read back successfully, and the rows match the original DataFrame. The only visible difference is that missing `last_review` values appear as `None` (SQL NULL) instead of `NaN`.

### 2.2 Filtering and Sorting

This query finds entire homes/apartments in Manhattan priced under \$150, excluding any listing with a price of 0, and shows the ten with the most reviews.

In [ ]:
# Entire homes in Manhattan priced under $150 (zero-price rows excluded),
# showing the ten with the most reviews
query_manhattan_homes = """
    SELECT name, neighbourhood, price, number_of_reviews
    FROM listings
    WHERE neighbourhood_group = 'Manhattan'
      AND room_type = 'Entire home/apt'
      AND price BETWEEN 1 AND 149
    ORDER BY number_of_reviews DESC
    LIMIT 10
"""
manhattan_homes = pd.read_sql_query(query_manhattan_homes, conn)
manhattan_homes

*[Run this cell, then add your own observation here.]*

## ***3. SQL on a DataFrame (`pandasql`)***

### Price Bands with `CASE`

`pandasql` runs SQL directly on a DataFrame, with no database connection or table creation. The `CASE` expression below sorts each listing into a Budget, Mid-range or Premium band, and the query counts the listings in each band and averages their yearly availability.

In [ ]:
!pip install pandasql

from pandasql import sqldf

# Split listings into price bands, then compare the size and the
# average yearly availability of each band
query_bands = """
    SELECT
        CASE
            WHEN price < 100 THEN 'Budget (under $100)'
            WHEN price < 250 THEN 'Mid-range ($100-$249)'
            ELSE 'Premium ($250+)'
        END AS price_band,
        COUNT(*) AS listings,
        ROUND(AVG(availability_365), 1) AS avg_days_available
    FROM df
    GROUP BY price_band
    ORDER BY MIN(price)
"""
price_bands = sqldf(query_bands)
price_bands

*[Run this cell, then add your own observation here.]*

## ***4. Aggregation and Ranking***

### 4.1 Average Price by Neighbourhood Group and the Top 5 Most Expensive Listings

The first query averages the price in each borough with `AVG` and `ROUND`, sorted from highest to lowest. The second uses `ORDER BY ... DESC` with `LIMIT 5` to list the most expensive listings.

In [6]:
# 1. Average price by neighbourhood group using SQL
query_avg_price = """
    SELECT neighbourhood_group, ROUND(AVG(price), 2) as avg_price
    FROM listings
    GROUP BY neighbourhood_group
    ORDER BY avg_price DESC
"""
avg_price_result = pd.read_sql_query(query_avg_price, conn)
print("Average Price by Neighbourhood Group:")
display(avg_price_result)

# 2. Top 5 most expensive listings
query_expensive = """
    SELECT name, neighbourhood_group, room_type, price
    FROM listings
    ORDER BY price DESC
    LIMIT 5
"""
expensive_result = pd.read_sql_query(query_expensive, conn)
print("\nTop 5 Most Expensive Listings:")
display(expensive_result)

Average Price by Neighbourhood Group:


,neighbourhood_group,avg_price
0,Manhattan,196.88
1,Brooklyn,124.38
2,Staten Island,114.81
3,Queens,99.52
4,Bronx,87.50



Top 5 Most Expensive Listings:


,name,neighbourhood_group,room_type,price
0,Furnished room in Astoria apartment,Queens,Private room,10000
1,Luxury 1 bedroom apt. -stunning Manhattan views,Brooklyn,Entire home/apt,10000
2,1-BR Lincoln Center,Manhattan,Entire home/apt,10000
3,Spanish Harlem Apt,Manhattan,Entire home/apt,9999
4,"Quiet, Clean, Lit @ LES & Chinatown",Manhattan,Private room,9999


**Observation:**

* Manhattan has the highest average price (\$196.88), followed by Brooklyn (\$124.38), Staten Island (\$114.81), Queens (\$99.52) and the Bronx (\$87.50).
* The five most expensive listings are priced between \$9,999 and \$10,000 a night. Some of these look unusual, for example a private room in Astoria priced at \$10,000, and a listing whose name mentions Manhattan but whose borough is Brooklyn. They may be data-entry errors or placeholder prices and are worth checking before being used in further analysis.

**Note:** These averages include every listing. The dataset also contains some listings with a price of 0, and the query does not exclude them or the extreme prices above, so both affect the averages.

### 4.2 Neighbourhoods with the Highest Average Price

`HAVING COUNT(*) >= 100` keeps only neighbourhoods with at least 100 listings, so that a few unusual listings in a small neighbourhood cannot dominate the ranking.

In [ ]:
# Neighbourhoods with at least 100 listings, ranked by average price
query_pricey_areas = """
    SELECT neighbourhood,
           neighbourhood_group,
           COUNT(*) AS listings,
           ROUND(AVG(price), 2) AS avg_price
    FROM listings
    GROUP BY neighbourhood, neighbourhood_group
    HAVING COUNT(*) >= 100
    ORDER BY avg_price DESC
    LIMIT 10
"""
pricey_areas = pd.read_sql_query(query_pricey_areas, conn)
pricey_areas

*[Run this cell, then add your own observation here.]*

## ***5. Saving Changes and Closing the Connection***

`commit()` saves any changes to the database file and `close()` ends the connection. A new connection must be opened with `sqlite3.connect()` before querying the database again.

In [7]:
# Save/commit changes and close the SQLite database connection
conn.commit()
conn.close()

print("Database connection successfully closed!")

Database connection successfully closed!


# **Conclusion**

**Key Findings and Insights:**

1. **SQLite Storage:** The Airbnb dataset was stored in a SQLite database (`nyc_airbnb.db`) with `to_sql()` and queried back with `pd.read_sql_query()`.
2. **SQL Querying in Python:** Filtering (`WHERE`, `BETWEEN`), grouping (`GROUP BY`, `HAVING`), aggregation (`AVG`, `COUNT`) and sorting (`ORDER BY`, `LIMIT`) were all done in SQL. The average price by borough matches the EDA notebook.
3. **Querying a DataFrame with `pandasql`:** `sqldf` ran SQL, including a `CASE` expression, directly on the DataFrame without a database connection, which is convenient for quick checks.
4. **Business Insight:** Manhattan is the most expensive borough and the Bronx the cheapest. A handful of listings priced near \$10,000 should be reviewed before pricing conclusions are drawn.

### ***End of SQL Data Exploration***